# 📚 Project #20: Book Discovery Architecture (v2)
**Architect:** Kemal Demirbaş 🏰🚀 | **Project Series:** 20 of 21

[![Hugging Face Space](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Live%20Demo-blue)](https://huggingface.co/spaces/Ironside35/NextRead-Collaborative)
[![Python](https://img.shields.io/badge/Python-3.12-yellow)](https://www.python.org/)
[![Logic](https://img.shields.io/badge/Logic-Collaborative%20Filtering-green)](https://en.wikipedia.org/wiki/Collaborative_filtering)
[![Metric](https://img.shields.io/badge/Metric-Cosine%20Similarity-orange)](https://en.wikipedia.org/wiki/Cosine_similarity)

---

### 📖 Project Vision: The Spatial Reader
> *"A book is more than a collection of words; it is a coordinate in the library of human thought. To recommend the next masterpiece, we must calculate the spatial distance between reading souls."*

**Project Overview:** For the 20th entry in this marathon, the architecture has undergone a rigorous refactoring process. Moving away from flawed metadata heuristics, we have engineered a true **Collaborative Filtering Recommendation Engine**. By utilizing the highly-regarded **Goodbooks-10k dataset**, we map the relationships between books based entirely on millions of actual user interactions (ratings), eliminating subjective "Genre DNA" in favor of pure, unbiased behavioral data.

---

## 🧠 The Brain: Matrix Factorization & Spatial Logic
This engine identifies similarities by treating users as dimensions in a vast spatial plane. Instead of comparing what a book *is* (genre), it compares *how it is perceived* by the masses:

* **The User-Item Matrix:** Books are represented as rows, and users as columns. The rating a user gives to a book is the magnitude of the vector in that specific dimension.
* **Cosine Similarity:** To find how close two books are, we calculate the angle between their interaction vectors. A smaller angle (Cosine score closer to 1.0) means the books share highly similar audiences.
  
  $$\text{Cosine Similarity}(A, B) = \frac{A \cdot B}{||A|| ||B||}$$

---

## 🏗️ Senior Pipeline (The Architectural Blueprint)
1.  **Cloud Ingestion:** Streaming the raw `goodbooks-10k` (Ratings and Books) datasets directly from stable GitHub repositories.
2.  **Noise Reduction (The Senior Filter):** Preventing memory leaks and sparse data issues by rigorously filtering the dataset. Only "Active Users" (>100 ratings) and "Popular Books" (>50 ratings) are permitted into the matrix.
3.  **Matrix Construction:** Building a robust Pivot Table (User-Item Matrix) where missing interactions are neutralized (filled with 0).
4.  **Vector Proximity:** Executing the `cosine_similarity` function across the entire multi-dimensional matrix to establish the distance between all literary pairs.
5.  **Inference Engine:** A custom function that locates a target book's row index, sorts its neighbors by proximity, and returns the Top 5 closest spatial matches (excluding the 100% self-match) with realistic percentage scores.
6.  **MLOps Persistence:** Sealing the calculated pivot tables and similarity matrices using `joblib` for rapid, low-latency deployment on edge servers (Hugging Face).

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import joblib

In [3]:
# ==============================================================================
# --- STEP 1: DATA INGESTION (GOODBOOKS-10K) ---
# ==============================================================================
print("--- 📂 Fetching Goodbooks-10k Datasets from GitHub ---")
# Fetching both ratings and books data directly from the raw GitHub URLs
ratings_url = 'https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv'
books_url = 'https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv'

--- 📂 Fetching Goodbooks-10k Datasets from GitHub ---


In [4]:
ratings = pd.read_csv(ratings_url)
books = pd.read_csv(books_url)

In [5]:
# Merging the ratings with book titles using 'book_id'
# We only extract 'book_id' and 'title' from the books dataset to save memory
book_data = ratings.merge(books[['book_id', 'title']], on='book_id')

In [6]:
# ==============================================================================
# --- STEP 2: MATRIX FACTORIZATION & NOISE REDUCTION ---
# ==============================================================================
print("--- ⚙️ Building User-Item Matrix ---")
# Filtering out noise:
# 1. Selecting users who have rated more than 100 books (Active readers)
user_counts = book_data.groupby('user_id').count()['rating'] > 100
experienced_users = user_counts[user_counts].index
filtered_rating = book_data[book_data['user_id'].isin(experienced_users)]

# 2. Selecting books that have received more than 50 ratings from these active users
book_counts = filtered_rating.groupby('title').count()['rating'] >= 50
famous_books = book_counts[book_counts].index
final_ratings = filtered_rating[filtered_rating['title'].isin(famous_books)]

# THE ARCHITECTURE: Pivot Matrix (Rows: Book Titles, Columns: User IDs, Values: Ratings)
pivot_table = final_ratings.pivot_table(index='title', columns='user_id', values='rating')
pivot_table.fillna(0, inplace=True) # Filling unrated entries with zero (0)

--- ⚙️ Building User-Item Matrix ---


In [7]:
# ==============================================================================
# --- STEP 3: SPATIAL DISTANCE (COSINE SIMILARITY) ---
# ==============================================================================
print("--- 📐 Calculating Vector Distances (Cosine Similarity) ---")
# Calculating the distance between books based on the angle of user rating vectors
similarity_scores = cosine_similarity(pivot_table)

--- 📐 Calculating Vector Distances (Cosine Similarity) ---


In [8]:
# ==============================================================================
# --- STEP 4: RECOMMENDATION ENGINE (INFERENCE) ---
# ==============================================================================
def recommend_book(book_name):
    try:
        # Finding the row number (index) of the book in the matrix
        index = np.where(pivot_table.index == book_name)[0][0]

        # Fetching the distance scores for that row and sorting them in descending order
        # [1:6] skips the first item (the book itself, 100% match) and gets the top 5
        similar_items = sorted(list(enumerate(similarity_scores[index])), key=lambda x: x[1], reverse=True)[1:6]

        print(f"\n📚 Vector-based Recommendations for readers of '{book_name}':")
        print("-" * 50)
        for i in similar_items:
            recommended_book = pivot_table.index[i[0]]
            # Converting similarity score to percentage
            similarity_percentage = round(i[1] * 100, 2)
            print(f"👉 {recommended_book} (Similarity Score: {similarity_percentage}%)")

    except IndexError:
        print(f"\n❌ '{book_name}' was not found in the library or lacks sufficient ratings. Try another book.")

In [13]:
recommend_book('The Hunger Games (The Hunger Games, #1)')


📚 Vector-based Recommendations for readers of 'The Hunger Games (The Hunger Games, #1)':
--------------------------------------------------
👉 Catching Fire (The Hunger Games, #2) (Similarity Score: 72.91%)
👉 Mockingjay (The Hunger Games, #3) (Similarity Score: 69.36%)
👉 Harry Potter and the Sorcerer's Stone (Harry Potter, #1) (Similarity Score: 60.67%)
👉 Twilight (Twilight, #1) (Similarity Score: 57.44%)
👉 Divergent (Divergent, #1) (Similarity Score: 52.18%)


In [9]:
# ==============================================================================
# --- STEP 5: MLOps (MODEL PERSISTENCE) ---
# ==============================================================================
joblib.dump(pivot_table, 'collaborative_pivot.pkl')
joblib.dump(similarity_scores, 'collaborative_similarity.pkl')


['collaborative_similarity.pkl']

## 📈 Audit & Success Report
Project #20 has been successfully fortified and sealed under the rigorous discipline of **Collaborative Filtering**. The system's integrity was verified through the following technical audits:

* **Behavioral Transparency:** By abandoning subjective "Genre DNA" heuristics and transitioning to a **User-Item Matrix**, the engine's recommendations are now 100% data-driven. The system relies on actual mass human behavior (ratings) rather than manual assumptions, ensuring unbiased and highly relevant suggestions.
* **Memory-Optimized Architecture:** Traditional pivot tables often cause out-of-memory (OOM) errors due to high sparsity. By engineering a strict "Noise Reduction" pipeline (filtering for Active Users >100 ratings and Established Books >50 ratings), we successfully constructed a dense, highly potent matrix that maximizes vector precision without crashing the server's RAM.
* **Deployment Readiness:** The persistent compilation of `collaborative_pivot.pkl` and `collaborative_similarity.pkl` via Joblib guarantees a low-latency, stateful transition to the live Hugging Face edge deployment.

## 🚀 Live Discovery Platform
Experience the 20th fortress in real-time here:
👉 **[Hugging Face Live Demo: Book Discovery Architecture](https://huggingface.co/spaces/Ironside35/NextRead-Collaborative)**

---
*Next Stop: Project #21 - THE GRAND FINALE 👑🏗️*